# 03 · Cox model and the cloglog GLM

Run each cell with **Shift+Enter**. Cells under **Your turn** contain `...` placeholders; replace them, run, then run the check beneath. Checks print `PASS` or `NOT YET` and never stop the notebook. Worked answers are in the matching `_solutions` notebook.

The checks compare your results with the **truth in `python/simulate.py`**, so they keep working if you change the simulator's settings.

The headline test of Milestone 3: **does the model recover the effects you built in?** A Cox model estimates log hazard ratios; your simulator's `TRUE_EFFECTS` are log hazard ratios. They should line up.

In [ ]:
def check(label, get, expected=True):
    """Print PASS or NOT YET. `get` is a lambda, so missing names or errors never stop the notebook."""
    try:
        got = get()
    except Exception as e:
        print(f"NOT YET  {label}: {type(e).__name__}: {e}")
        return
    try:
        ok = got.equals(expected) if hasattr(got, "equals") else bool(got == expected)
        if isinstance(expected, float) and isinstance(got, (int, float)):
            ok = abs(got - expected) < 1e-6
    except Exception:
        ok = False
    print(f"PASS     {label}" if ok else f"NOT YET  {label}: expected {expected!r}, got {got!r}")

In [ ]:
import sys
sys.path.insert(0, "../python")

import importlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import simulate as sim
importlib.reload(sim)

members = pd.read_parquet("../data/members.parquet")
print(f"{len(members):,} members, {members['lapsed'].mean():.1%} lapsed")

from lifelines import CoxPHFitter
from lifelines.statistics import proportional_hazard_test

def design(df):
    """Dummy columns named like TRUE_EFFECTS (reference levels dropped), plus premium_increase."""
    X = pd.DataFrame(index=df.index)
    for col, (levels, _) in sim.LEVELS.items():
        for level in levels[1:]:
            X[f"{col}[{level}]"] = (df[col] == level).astype(float)
    X["premium_increase"] = df["premium_increase"]
    return X

## 1. Fit the Cox model

### Your turn 1.1

Fit `cph` on `design(members)` plus the `duration` and `lapsed` columns. Then build `results`: one row per effect with columns `truth` (from `sim.TRUE_EFFECTS`), `estimate`, `lower` and `upper` (95% interval), and `hazard_ratio`.

Hints: `cph.params_` holds the estimates, `cph.confidence_intervals_` the bounds (two columns), and `pd.Series(sim.TRUE_EFFECTS)` the truth.

In [ ]:
cox_data = design(members).assign(duration=members["duration"], lapsed=members["lapsed"])
cph = ...
results = ...

In [ ]:
check("1.1 every effect within 0.1 of the truth", lambda: (results["estimate"] - results["truth"]).abs().max() < 0.1)
check("1.1 most truths inside the 95% interval", lambda: ((results["truth"] >= results["lower"]) & (results["truth"] <= results["upper"])).sum() >= 6)

## 2. Proportional hazards

Cox assumes each effect multiplies the hazard by the same factor at every tenure. Your simulator builds that in, so this test should mostly pass. The anniversary spikes do **not** break it: they live in the baseline hazard, which Cox leaves unrestricted. On real data, this is where you would find effects that fade or grow with tenure.

In [ ]:
if hasattr(cph, "params_"):
    ph = proportional_hazard_test(cph, cox_data, time_transform="rank")
    display(ph.summary[["test_statistic", "p"]].round(4))
else:
    print("Fit the Cox model in section 1 first.")

## 3. The model that matches the simulator exactly

Your simulator is a binomial model on person-months with a cloglog link. Fitting that model directly recovers **everything**: the member effects, the baseline and the tenure effects.

### Your turn 3.1

Build `pm` with `sim.person_months(members)` and fit `glm_cll`:

```text
lapse ~ C(age_band) + C(product_tier) + C(channel, Treatment('direct')) + premium_increase + early + anniversary + long
```

with `family=sm.families.Binomial(link=sm.families.links.CLogLog())`. It takes 20 to 60 seconds on over a million rows.

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

pm = ...
glm_cll = ...

In [ ]:
check("3.1 baseline recovered", lambda: abs(glm_cll.params["Intercept"] - sim.BASELINE) < 0.1)
check("3.1 anniversary effect recovered", lambda: abs(glm_cll.params["anniversary"] - sim.TENURE_EFFECTS["anniversary"]) < 0.1)
check("3.1 early effect recovered", lambda: abs(glm_cll.params["early"] - sim.TENURE_EFFECTS["early"]) < 0.1)
check("3.1 long effect recovered", lambda: abs(glm_cll.params["long"] - sim.TENURE_EFFECTS["long"]) < 0.1)
check("3.1 premium effect recovered", lambda: abs(glm_cll.params["premium_increase"] - sim.TRUE_EFFECTS["premium_increase"]) < 0.02)

## 4. Check the Cox model against R

Save the data, run `r/cox_check.R` from the repository root (`Rscript r/cox_check.R` in the VS Code terminal), then rerun the next cell. Both use the Efron method for tied durations, so the estimates should agree closely. R names a dummy `age_band31-45`; the cell converts that to `age_band[31-45]` and matches by name.

In [ ]:
members.to_csv("../data/members.csv", index=False)
print("Saved data/members.csv. Now run: Rscript r/cox_check.R")

In [ ]:
import os
import re

if os.path.exists("../r_outputs/cox_coefs.csv"):
    r_cox = pd.read_csv("../r_outputs/cox_coefs.csv")
    r_cox["term"] = r_cox["term"].str.replace(r"^(age_band|product_tier|channel)(.+)$", r"\1[\2]", regex=True)
    r_cox = r_cox.set_index("term")["estimate"]
    if hasattr(cph, "params_"):
        print(pd.DataFrame({"python": cph.params_, "r": r_cox}).round(5))
        check("4 Cox matches R", lambda: np.allclose(cph.params_, r_cox.reindex(cph.params_.index), atol=1e-4))
else:
    print("Run Rscript r/cox_check.R first, then rerun this cell.")

## Wrap-up

Record the recovery table in your notes: it is the centrepiece of the README. Commit from VS Code: `Cox and cloglog GLM recover the true effects`.